# Semana 2: Preprocesamiento de Texto
## Notebook de Ejercicios (NB2) – Preprocesamiento de Reseñas de Películas

**Propósito:** Aplicar un pipeline completo de preprocesamiento a un corpus real de reseñas de películas (IMDb) y analizar el impacto en las palabras más frecuentes.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Aplicar un pipeline completo de preprocesamiento: limpieza con regex, tokenización, eliminación de stopwords, lematización.
- Crear una función reutilizable que normalice cualquier texto nuevo.
- Visualizar las palabras más frecuentes antes y después del preprocesamiento.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y descargamos los recursos lingüísticos requeridos.

In [ ]:
# Importamos librerías
import re
import nltk
import spacy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm

# NLTK recursos
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# spaCy
try:
    nlp = spacy.load('en_core_web_sm')
    print("Modelo de spaCy cargado correctamente.")
except:
    print("Descargando modelo de spaCy...")
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

# Hugging Face datasets
try:
    from datasets import load_dataset
    DATASETS_AVAILABLE = True
except ImportError:
    DATASETS_AVAILABLE = False
    print("Nota: datasets no está instalado. Se instalará más adelante.")

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("\nLibrerías importadas correctamente.")

---
## 1. Carga del Dataset IMDb Reviews

El dataset **IMDb reviews** contiene 50,000 reseñas de películas etiquetadas como positivas o negativas. Para este ejercicio, nos enfocaremos en el preprocesamiento, por lo que trabajaremos con una muestra de reseñas.

In [ ]:
if not DATASETS_AVAILABLE:
    !pip install datasets
    from datasets import load_dataset

# Cargamos el dataset IMDb
print("Cargando dataset IMDb...")
imdb = load_dataset('imdb')

# Tomamos una muestra de 1000 reseñas para el ejercicio
sample_size = 1000
train_reviews = imdb['train']['text'][:sample_size]
train_labels = imdb['train']['label'][:sample_size]

print(f"Muestra cargada: {len(train_reviews)} reseñas.")
print("\nEjemplo de reseña:")
print(train_reviews[0][:500] + "...")

### 1.1. Exploración inicial de las reseñas

In [ ]:
# Estadísticas básicas
review_lengths = [len(review.split()) for review in train_reviews]

print(f"Longitud promedio de reseñas: {np.mean(review_lengths):.0f} palabras")
print(f"Longitud mínima: {min(review_lengths)} palabras")
print(f"Longitud máxima: {max(review_lengths)} palabras")

plt.figure(figsize=(10, 4))
plt.hist(review_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Número de palabras')
plt.ylabel('Frecuencia')
plt.title('Distribución de longitud de reseñas (antes del preprocesamiento)')
plt.show()

---
## 2. Pipeline de Preprocesamiento

Diseñamos un pipeline completo que incluye:
1. Limpieza con expresiones regulares
2. Tokenización
3. Eliminación de stopwords
4. Lematización

In [ ]:
def clean_text(text):
    """Limpieza básica con regex."""
    # Eliminar HTML tags (comunes en IMDb)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Eliminar URLs
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text)
    
    # Eliminar menciones y hashtags (aunque no son comunes en IMDb)
    text = re.sub(r'@\S+', '<USER>', text)
    text = re.sub(r'#\S+', '<HASHTAG>', text)
    
    # Eliminar números
    text = re.sub(r'\d+', '<NUM>', text)
    
    # Eliminar caracteres especiales y puntuación (mantener letras y espacios)
    text = re.sub(r'[^\w\s<>]', ' ', text)
    
    # Normalizar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Convertir a minúsculas
    text = text.lower()
    
    return text

In [ ]:
def preprocess_review(text, remove_stopwords=True, use_lemmatization=True):
    """
    Pipeline completo de preprocesamiento para una reseña.
    """
    # 1. Limpieza con regex
    text = clean_text(text)
    
    # 2. Tokenización con spaCy
    doc = nlp(text)
    
    tokens = []
    for token in doc:
        # Verificar si es stopword (opcional)
        if remove_stopwords and token.is_stop:
            continue
        
        # Verificar si es puntuación o espacio
        if token.is_punct or token.is_space:
            continue
        
        # Aplicar lematización
        if use_lemmatization:
            tokens.append(token.lemma_)
        else:
            tokens.append(token.text)
    
    return tokens

# Probamos la función con una reseña
print("=== Prueba del pipeline ===")
test_review = train_reviews[0]
print("\nTexto original:")
print(test_review[:300])

processed = preprocess_review(test_review)
print("\nTexto preprocesado (primeros 50 tokens):")
print(processed[:50])

---
## 3. Aplicación del Pipeline a la Muestra

Procesamos todas las reseñas de la muestra. Esto puede tomar unos minutos.

In [ ]:
# Procesamos todas las reseñas
print("Procesando reseñas...")
processed_reviews = []
for review in tqdm(train_reviews):
    processed = preprocess_review(review)
    processed_reviews.append(processed)

print(f"Procesamiento completado. {len(processed_reviews)} reseñas procesadas.")

### 3.1. Estadísticas después del preprocesamiento

In [ ]:
processed_lengths = [len(tokens) for tokens in processed_reviews]

print(f"Longitud promedio después del preprocesamiento: {np.mean(processed_lengths):.0f} tokens")
print(f"Reducción promedio: {np.mean(review_lengths) - np.mean(processed_lengths):.0f} tokens")

plt.figure(figsize=(10, 4))
plt.hist(processed_lengths, bins=50, edgecolor='black', alpha=0.7, color='green')
plt.xlabel('Número de tokens')
plt.ylabel('Frecuencia')
plt.title('Distribución de longitud de reseñas (después del preprocesamiento)')
plt.show()

---
## 4. Visualización de Palabras Más Frecuentes

Comparamos las palabras más frecuentes antes y después del preprocesamiento.

### 4.1. Palabras más frecuentes ANTES del preprocesamiento

In [ ]:
# Unimos todas las reseñas originales en un solo texto
all_text_before = ' '.join(train_reviews)

# Tokenización simple (sin procesar) para contar frecuencias
tokens_before = word_tokenize(all_text_before.lower())

# Contar frecuencias
word_freq_before = Counter(tokens_before)

# Mostrar las 20 palabras más frecuentes (ignorando puntuación)
common_before = word_freq_before.most_common(30)

# Filtrar puntuación para visualización
filtered_before = [(word, freq) for word, freq in common_before if word.isalpha()][:20]

words_before, freqs_before = zip(*filtered_before)

plt.figure(figsize=(12, 6))
plt.bar(words_before, freqs_before)
plt.xlabel('Palabra')
plt.ylabel('Frecuencia')
plt.title('Top 20 palabras más frecuentes - ANTES del preprocesamiento')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Palabras más frecuentes (antes):")
for word, freq in filtered_before[:10]:
    print(f"{word:15} {freq:6,} apariciones")

### 4.2. Palabras más frecuentes DESPUÉS del preprocesamiento

In [ ]:
# Unimos todos los tokens procesados
all_tokens_after = [token for review in processed_reviews for token in review]

# Contar frecuencias
word_freq_after = Counter(all_tokens_after)

# Mostrar las 20 palabras más frecuentes
common_after = word_freq_after.most_common(20)

words_after, freqs_after = zip(*common_after)

plt.figure(figsize=(12, 6))
plt.bar(words_after, freqs_after, color='green')
plt.xlabel('Palabra')
plt.ylabel('Frecuencia')
plt.title('Top 20 palabras más frecuentes - DESPUÉS del preprocesamiento')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Palabras más frecuentes (después):")
for word, freq in common_after[:10]:
    print(f"{word:15} {freq:6,} apariciones")

### 4.3. Comparación visual con nubes de palabras (Word Clouds)

Instalamos wordcloud si no está disponible.

In [ ]:
try:
    from wordcloud import WordCloud
except ImportError:
    !pip install wordcloud
    from wordcloud import WordCloud

# Crear nubes de palabras
wordcloud_before = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(all_text_before)
wordcloud_after = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(' '.join(all_tokens_after))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(wordcloud_before, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Nube de palabras - ANTES del preprocesamiento')

axes[1].imshow(wordcloud_after, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Nube de palabras - DESPUÉS del preprocesamiento')

plt.tight_layout()
plt.show()

---
## 5. Función de Normalización para Textos Nuevos

Creamos una función que encapsula todo el pipeline y puede aplicarse a cualquier texto nuevo.

In [ ]:
def normalize_text(text, remove_stopwords=True, use_lemmatization=True):
    """
    Normaliza un texto aplicando todo el pipeline de preprocesamiento.
    
    Args:
        text (str): Texto a normalizar
        remove_stopwords (bool): Si se deben eliminar stopwords
        use_lemmatization (bool): Si se debe aplicar lematización
    
    Returns:
        list: Lista de tokens procesados
    """
    return preprocess_review(text, remove_stopwords, use_lemmatization)

# Probamos la función con algunas reseñas nuevas
test_texts = [
    "This movie was absolutely fantastic! I loved it 😊",
    "Terrible film, complete waste of time. Don't watch it!",
    "Check out https://example.com for more reviews #movies @filmcritic"
]

print("=== Prueba de la función de normalización ===\n")
for text in test_texts:
    print(f"Original: {text}")
    normalized = normalize_text(text)
    print(f"Normalizado: {normalized}")
    print()

### 5.1. Experimentación con diferentes parámetros

Probamos la función con diferentes configuraciones.

In [ ]:
test_review = train_reviews[1][:200]  # una reseña corta

configs = [
    (True, True, "Con stopwords y lematización"),
    (False, True, "Sin stopwords, con lematización"),
    (True, False, "Con stopwords, sin lematización"),
    (False, False, "Sin stopwords, sin lematización")
]

print(f"Texto original: {test_review}\n")

for remove_stop, use_lemma, desc in configs:
    result = normalize_text(test_review, remove_stopwords=remove_stop, use_lemmatization=use_lemma)
    print(f"{desc}:")
    print(f"  {result[:30]}...")
    print(f"  Longitud: {len(result)} tokens")
    print()

---
## 6. Análisis del Impacto del Preprocesamiento

Reflexionamos sobre cómo el preprocesamiento afecta al texto.

In [ ]:
print("=== ANÁLISIS DEL IMPACTO ===\n")

print(f"Número total de palabras antes: {len(tokens_before):,}")
print(f"Número total de tokens después: {len(all_tokens_after):,}")
print(f"Reducción: {(1 - len(all_tokens_after)/len(tokens_before))*100:.1f}%")

print(f"\nTamaño del vocabulario antes: {len(set(tokens_before)):,}")
print(f"Tamaño del vocabulario después: {len(set(all_tokens_after)):,}")
print(f"Reducción: {(1 - len(set(all_tokens_after))/len(set(tokens_before)))*100:.1f}%")

### Preguntas para reflexionar:

1. **¿Qué palabras desaparecieron después del preprocesamiento?**
   - Stopwords (the, a, and, etc.)
   - Variantes morfológicas (running, runs → run)
   - Puntuación y caracteres especiales

2. **¿Qué palabras ganaron importancia relativa?**
   - Palabras de contenido semántico (movie, film, good, bad, great, terrible)
   - Términos relevantes para el análisis de sentimiento

3. **¿Cómo afectaría esto a un clasificador de sentimiento?**
   - Reduciría el ruido y la dimensionalidad
   - Permitiría que el modelo se enfoque en palabras con carga semántica
   - Podría mejorar la precisión si se eliminan stopwords, pero cuidado con palabras como "not"

4. **¿En qué casos no convendría eliminar stopwords?**
   - En modelos basados en atención (Transformers) que pueden aprender a ignorarlas
   - Cuando la negación es importante ("not good" vs "good")
   - En análisis estilísticos o de autoría

---
## 7. Conclusiones

En este notebook hemos aplicado un pipeline completo de preprocesamiento a un corpus real de reseñas de películas:

✔️ **Pipeline de preprocesamiento**:
- Limpieza con regex (HTML, URLs, números, caracteres especiales)
- Tokenización con spaCy
- Eliminación de stopwords
- Lematización

✔️ **Función de normalización**: Creamos una función reutilizable que puede aplicarse a cualquier texto nuevo.

✔️ **Análisis de frecuencias**:
- Antes del preprocesamiento: predominan stopwords y palabras comunes
- Después del preprocesamiento: emergen palabras con carga semántica ("film", "movie", "good", "great")

✔️ **Impacto cuantitativo**:
- Reducción del número de tokens (~30-40%)
- Reducción del tamaño del vocabulario (~20-30%)

**Lección clave**: El preprocesamiento es esencial para reducir ruido y enfocar los modelos en la información relevante. Sin embargo, las decisiones (eliminar stopwords, lematizar) deben tomarse considerando la tarea final y el tipo de modelo a utilizar.

---
**Fin del Notebook de Ejercicios - Semana 2 (NLP)**